In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc, average_precision_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# Load data

In [2]:
df = pd.read_csv('Dataset/paysim_dataset_processed.csv')
df_tree = df.copy()
df_nonTree = df.copy()
df.head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,OrgbalanceDiff,DestbalanceDiff,hour_of_day,day_of_week,type_CASH_OUT,type_TRANSFER
0,181.00,181.0,0.0,0.0,0.00,1,-181.0,0.00,1,0,0.0,1.0
1,181.00,181.0,0.0,21182.0,0.00,1,-181.0,-21182.00,1,0,1.0,0.0
2,229133.94,15325.0,0.0,5083.0,51513.44,0,-15325.0,46430.44,1,0,1.0,0.0
3,215310.30,705.0,0.0,22425.0,0.00,0,-705.0,-22425.00,1,0,0.0,1.0
4,311685.89,10835.0,0.0,6267.0,2719172.89,0,-10835.0,2712905.89,1,0,0.0,1.0


# Preprocess

In [3]:
X_tree = df_tree.drop('isFraud', axis=1)
y_tree = df_tree['isFraud']

X_nonTree = df_nonTree.drop('isFraud', axis=1)
y_nonTree = df_nonTree['isFraud']

print(f"{X_tree.shape=}, {y_tree.shape=}")
print(f"{X_nonTree.shape=}, {y_nonTree.shape=}")

X_tree.shape=(2770409, 11), y_tree.shape=(2770409,)
X_nonTree.shape=(2770409, 11), y_nonTree.shape=(2770409,)


**dataset for tree base models**

In [4]:
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_tree, y_tree, test_size=0.2, random_state=42, stratify=y_tree, shuffle=True)
print(f"{Xt_train.shape=}, {yt_train.shape=}")
print(f"{Xt_test.shape=}, {yt_test.shape=}")

Xt_train.shape=(2216327, 11), yt_train.shape=(2216327,)
Xt_test.shape=(554082, 11), yt_test.shape=(554082,)


In [5]:
Xt_test.to_csv('Dataset/X_test.csv', index=False)
yt_test.to_csv('Dataset/y_test.csv', index=False)

**dataset for nonTree base models**

In [16]:
Xnt_train, Xnt_test, ynt_train, ynt_test = train_test_split(X_nonTree, y_nonTree, test_size=0.2, random_state=42, stratify=y_nonTree, shuffle=True)
print(f"{Xnt_train.shape=}, {ynt_train.shape=}")
print(f"{Xnt_test.shape=}, {ynt_test.shape=}")

Xnt_train.shape=(2216327, 11), ynt_train.shape=(2216327,)
Xnt_test.shape=(554082, 11), ynt_test.shape=(554082,)


# Train models

**Models use**

In [17]:
tree_models = {
    'decision_tree': DecisionTreeClassifier(random_state=42, max_depth=20),
    'random_forest': RandomForestClassifier(random_state=42, max_depth=20, n_estimators=100, n_jobs=-1),
    'xgboost': XGBClassifier(random_state=42, eval_metric='logloss', objective='binary:logistic', device="cuda"),
    'lightgbm': LGBMClassifier(random_state=42, max_depth =20, n_estimators=100, n_jobs=-1)
}

non_tree_models = {
    'logistic_regression': LogisticRegression(random_state=42, max_iter=1000),
    'knn': KNeighborsClassifier(),
    'naive_bayes': GaussianNB()
}

**Training nonTree-base models**

In [18]:
for name, model in non_tree_models.items():
    print(f"Training {name}")
    
    nonTreePipeline = Pipeline([
        ('transformer', PowerTransformer()),
        ('scaler', MinMaxScaler()),
        ('smote', SMOTE(random_state=42)),
        ('pca', PCA(n_components=0.95, random_state=42)),
        ('classifier', model)
    ])

    nonTreePipeline.fit(Xnt_train, ynt_train)
    ynt_pred = nonTreePipeline.predict(Xnt_test)
    ynt_probs = nonTreePipeline.predict_proba(Xnt_test)[:, 1] if hasattr(nonTreePipeline.named_steps['classifier'], 'predict_proba') else None
    precision, recall, _ = precision_recall_curve(ynt_test, ynt_probs) if ynt_probs is not None else (None, None, None)
    pr_auc = auc(recall, precision) if precision is not None and recall is not None else None
    
    print(f"Classification Report for {name}:\n{classification_report(ynt_test, ynt_pred)}")
    print(f"ROC AUC Score for {name}: {roc_auc_score(ynt_test, ynt_probs) if ynt_probs is not None else 'N/A'}")
    print(f"PR AUC Score for {name}: {pr_auc:.4f}")
    print(f"Confusion Matrix for {name}:\n{confusion_matrix(ynt_test, ynt_pred)}\n")

Training logistic_regression
Classification Report for logistic_regression:
              precision    recall  f1-score   support

           0       1.00      0.88      0.94    552439
           1       0.02      0.91      0.04      1643

    accuracy                           0.88    554082
   macro avg       0.51      0.90      0.49    554082
weighted avg       1.00      0.88      0.93    554082

ROC AUC Score for logistic_regression: 0.9677429612014227
PR AUC Score for logistic_regression: 0.5871
Confusion Matrix for logistic_regression:
[[486969  65470]
 [   145   1498]]

Training knn
Classification Report for knn:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    552439
           1       0.33      0.85      0.47      1643

    accuracy                           0.99    554082
   macro avg       0.66      0.92      0.73    554082
weighted avg       1.00      0.99      1.00    554082

ROC AUC Score for knn: 0.9323539131389568
PR 

**Tree-base Models**

In [19]:
for name, model in tree_models.items():
    print(f"Training {name}")
    
    treePipeline = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    treePipeline.fit(Xt_train, yt_train)

    yt_pred = treePipeline.predict(Xt_test)
    yt_probs = treePipeline.predict_proba(Xt_test)[:, 1]

    precision, recall, _ = precision_recall_curve(yt_test, yt_probs)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(yt_test, yt_probs)

    print(f"Classification Report for {name}:\n{classification_report(yt_test, yt_pred)}")
    print(f"ROC AUC Score for {name}: {roc_auc:.4f}")
    print(f"PR AUC Score for {name}: {pr_auc:.4f}")
    print(f"Confusion Matrix for {name}:\n{confusion_matrix(yt_test, yt_pred)}\n")
    print("=" * 50)

Training decision_tree
Classification Report for decision_tree:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.60      0.95      0.73      1643

    accuracy                           1.00    554082
   macro avg       0.80      0.97      0.87    554082
weighted avg       1.00      1.00      1.00    554082

ROC AUC Score for decision_tree: 0.9737
PR AUC Score for decision_tree: 0.8485
Confusion Matrix for decision_tree:
[[551382   1057]
 [    85   1558]]

Training random_forest
Classification Report for random_forest:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.56      0.96      0.71      1643

    accuracy                           1.00    554082
   macro avg       0.78      0.98      0.85    554082
weighted avg       1.00      1.00      1.00    554082

ROC AUC Score for random_forest: 0.9981
PR AUC Score for random_for

We chose the XGBoost model for hyperparameter tuning because it has a higher recall rate than most other models and a lower false positive rate than LightGBM. LightGBM, despite its high recall rate, has a much higher false positive rate than XGBoost. While the random forest model also performs well, it has a higher false positive rate than XGBoost. In this problem, we prioritize correctly capturing as many unusual transactions as possible, so we chose the XGBoost model.

# Tuning Hyperparameter

In [21]:
grid_params = {
    'classifier__n_estimators': [100, 300, 500, 700],
    'classifier__max_depth': [5, 6, 7, 8],
    'classifier__min_child_weight': [1, 3, 5],
    'classifier__gamma': [0, 0.1, 0.5, 1],
    'classifier__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'classifier__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'classifier__colsample_bylevel': [0.5, 0.7, 1.0],
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'classifier__reg_alpha': [0, 0.01, 0.1, 0.5, 1],
    'classifier__reg_lambda': [0.5, 1, 2, 5, 10]
}

xgboostPipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss', objective='binary:logistic', device="cuda"))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(estimator=xgboostPipeline, param_distributions=grid_params, n_iter=50, cv=cv, scoring='average_precision', random_state=42, verbose=2)

random_search.fit(Xt_train, yt_train)
best_model = random_search.best_estimator_
y_pred = best_model.predict(Xt_test)

print(f"Best Parameters: {random_search.best_params_}")
print(f"Best PR AUC Score: {random_search.best_score_:.4f}")
print(f"Classification Report for XGBoost with Best Parameters:\n{classification_report(yt_test, y_pred)}")
print(f"Confusion Matrix for XGBoost with Best Parameters:\n{confusion_matrix(yt_test, y_pred)}\n")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END classifier__colsample_bylevel=0.5, classifier__colsample_bytree=0.6, classifier__gamma=0, classifier__learning_rate=0.01, classifier__max_depth=6, classifier__min_child_weight=1, classifier__n_estimators=700, classifier__reg_alpha=0.5, classifier__reg_lambda=1, classifier__subsample=0.9; total time= 1.2min
[CV] END classifier__colsample_bylevel=0.5, classifier__colsample_bytree=0.6, classifier__gamma=0, classifier__learning_rate=0.01, classifier__max_depth=6, classifier__min_child_weight=1, classifier__n_estimators=700, classifier__reg_alpha=0.5, classifier__reg_lambda=1, classifier__subsample=0.9; total time= 1.1min
[CV] END classifier__colsample_bylevel=0.5, classifier__colsample_bytree=0.6, classifier__gamma=0, classifier__learning_rate=0.01, classifier__max_depth=6, classifier__min_child_weight=1, classifier__n_estimators=700, classifier__reg_alpha=0.5, classifier__reg_lambda=1, classifier__subsample=0.9; total 

**Save Best Model**

In [22]:
joblib.dump(best_model, 'best_xgboost_model.pkl')
print("Best XGBoost model saved as 'best_xgboost_model.pkl'")

Best XGBoost model saved as 'best_xgboost_model.pkl'
